In [ ]:
pip install wfdb

In [ ]:
pip install xgboost

In [ ]:
pip install PyWavelets

Import Libraries

In [6]:
import os
import wfdb
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, balanced_accuracy_score, f1_score, roc_auc_score

Baseline

In [9]:
# 1. Load metadata
df = pd.read_csv("C:\Kuliah\Lomba\IDSC\Misc\metadata.csv")

# 2. Keep binary only
df = df[df["brugada"].isin([0, 1])].copy()
df["label"] = df["brugada"].astype(int)

print(df["label"].value_counts())
print("Total samples:", len(df))

# 3. ECG loader
def load_ecg_record(patient_id, base_path=r"C:\Kuliah\Lomba\IDSC\Misc\files"):
    patient_id = str(patient_id)
    record_path = os.path.join(base_path, patient_id, patient_id)
    record = wfdb.rdrecord(record_path)
    return record.p_signal

# 4. Feature extraction
def extract_features(signal):
    feats = []
    for i in range(signal.shape[1]):
        x = signal[:, i]
        feats.extend([
            np.mean(x),
            np.std(x),
            np.min(x),
            np.max(x),
            np.median(x),
            np.sum(x**2),
            skew(x),
            kurtosis(x)
        ])
    return np.array(feats, dtype=float)

# 5. Build X, y
X, y, ids = [], [], []

for _, row in df.iterrows():
    pid = row["patient_id"]
    try:
        signal = load_ecg_record(pid)
        feats = extract_features(signal)

        X.append(feats)
        y.append(row["label"])
        ids.append(pid)
    except Exception as e:
        print(f"Failed {pid}: {e}")

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

# 6. Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 7. Train baseline
model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)
model.fit(X_train, y_train)

# 8. Evaluate
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]

print("Balanced Accuracy:", balanced_accuracy_score(y_test, pred))
print("F1 Score:", f1_score(y_test, pred))
print("ROC AUC:", roc_auc_score(y_test, prob))
print(classification_report(y_test, pred))

<>:2: SyntaxWarning: invalid escape sequence '\K'
<>:2: SyntaxWarning: invalid escape sequence '\K'
C:\Users\Owner\AppData\Local\Temp\ipykernel_21716\3834872471.py:2: SyntaxWarning: invalid escape sequence '\K'
  df = pd.read_csv("C:\Kuliah\Lomba\IDSC\Misc\metadata.csv")


label
0    287
1     69
Name: count, dtype: int64
Total samples: 356
X shape: (356, 96)
y shape: (356,)
Balanced Accuracy: 0.5
F1 Score: 0.0
ROC AUC: 0.7610837438423645
              precision    recall  f1-score   support

           0       0.81      1.00      0.89        58
           1       0.00      0.00      0.00        14

    accuracy                           0.81        72
   macro avg       0.40      0.50      0.45        72
weighted avg       0.65      0.81      0.72        72



c:\Users\Owner\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Owner\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Owner\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

Improvement Model

In [10]:
import numpy as np

print("Min prob:", prob.min())
print("Max prob:", prob.max())
print("Mean prob:", prob.mean())

print("Top 20 probs:", np.sort(prob)[-20:])

Min prob: 0.0033333333333333335
Max prob: 0.47333333333333333
Mean prob: 0.16870370370370372
Top 20 probs: [0.22333333 0.23333333 0.24       0.25       0.25666667 0.26
 0.28333333 0.28333333 0.28666667 0.29       0.29333333 0.30333333
 0.32333333 0.34333333 0.36333333 0.38       0.39333333 0.39666667
 0.40666667 0.47333333]


In [11]:
from sklearn.metrics import f1_score, balanced_accuracy_score

thresholds = np.arange(0.1, 0.9, 0.05)

best_t = None
best_f1 = -1

for t in thresholds:
    pred_t = (prob >= t).astype(int)
    f1 = f1_score(y_test, pred_t, zero_division=0)
    bacc = balanced_accuracy_score(y_test, pred_t)
    print(f"Threshold={t:.2f} | F1={f1:.4f} | Balanced Acc={bacc:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print("Best threshold:", best_t)
print("Best F1:", best_f1)

Threshold=0.10 | F1=0.4000 | Balanced Acc=0.6367
Threshold=0.15 | F1=0.4800 | Balanced Acc=0.7217
Threshold=0.20 | F1=0.4651 | Balanced Acc=0.6933
Threshold=0.25 | F1=0.4000 | Balanced Acc=0.6281
Threshold=0.30 | F1=0.4348 | Balanced Acc=0.6441
Threshold=0.35 | F1=0.3000 | Balanced Acc=0.5813
Threshold=0.40 | F1=0.1250 | Balanced Acc=0.5271
Threshold=0.45 | F1=0.1333 | Balanced Acc=0.5357
Threshold=0.50 | F1=0.0000 | Balanced Acc=0.5000
Threshold=0.55 | F1=0.0000 | Balanced Acc=0.5000
Threshold=0.60 | F1=0.0000 | Balanced Acc=0.5000
Threshold=0.65 | F1=0.0000 | Balanced Acc=0.5000
Threshold=0.70 | F1=0.0000 | Balanced Acc=0.5000
Threshold=0.75 | F1=0.0000 | Balanced Acc=0.5000
Threshold=0.80 | F1=0.0000 | Balanced Acc=0.5000
Threshold=0.85 | F1=0.0000 | Balanced Acc=0.5000
Best threshold: 0.15000000000000002
Best F1: 0.48


In [12]:
best_pred = (prob >= best_t).astype(int)

from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, best_pred))
print(classification_report(y_test, best_pred, zero_division=0))

[[34 24]
 [ 2 12]]
              precision    recall  f1-score   support

           0       0.94      0.59      0.72        58
           1       0.33      0.86      0.48        14

    accuracy                           0.64        72
   macro avg       0.64      0.72      0.60        72
weighted avg       0.83      0.64      0.68        72



In [13]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

lr_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=3000, class_weight="balanced")
)

lr_model.fit(X_train, y_train)
lr_prob = lr_model.predict_proba(X_test)[:, 1]

In [14]:
def extract_features_v2(signal):
    feats = []
    for i in range(signal.shape[1]):
        x = signal[:, i]
        feats.extend([
            np.mean(x),
            np.std(x),
            np.min(x),
            np.max(x),
            np.max(x) - np.min(x),
            np.median(x),
            np.percentile(x, 5),
            np.percentile(x, 25),
            np.percentile(x, 75),
            np.percentile(x, 95),
            np.mean(np.abs(x)),
            np.sqrt(np.mean(x**2)),
            np.sum(x**2),
            skew(x),
            kurtosis(x),
        ])
    return np.array(feats, dtype=float)

In [17]:
signal, lead_names, fs = load_ecg_record(df.iloc[0]["patient_id"]), None, None
def load_ecg_record_full(patient_id, base_path=r"C:\Kuliah\Lomba\IDSC\Misc\files"):
    patient_id = str(patient_id)
    record_path = os.path.join(base_path, patient_id, patient_id)
    record = wfdb.rdrecord(record_path)
    return record.p_signal, record.sig_name, record.fs

signal, lead_names, fs = load_ecg_record_full(df.iloc[0]["patient_id"])
print(lead_names)

['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']


In [18]:
signal_v123 = signal[:, [6, 7, 8]]

In [19]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, f1_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=3000, class_weight="balanced")
)

oof_prob = cross_val_predict(
    lr_model, X, y, cv=skf, method="predict_proba"
)[:, 1]

print("OOF ROC AUC:", roc_auc_score(y, oof_prob))

OOF ROC AUC: 0.6419229409685402


Once

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import wfdb

from scipy.stats import skew, kurtosis

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)

warnings.filterwarnings("ignore")


# =========================
# 1. CONFIG
# =========================
METADATA_PATH = "C:\Kuliah\Lomba\IDSC\Misc\metadata.csv"
FILES_DIR = r"C:\Kuliah\Lomba\IDSC\Misc\files"
RANDOM_STATE = 42
TEST_SIZE = 0.2


# =========================
# 2. LOAD METADATA
# =========================
print("=" * 60)
print("STEP 1 - LOAD METADATA")
print("=" * 60)

df = pd.read_csv(METADATA_PATH)

print("Original shape:", df.shape)
print("Original brugada distribution:")
print(df["brugada"].value_counts(dropna=False))
print()

# Keep only binary classes: 0 vs 1
df = df[df["brugada"].isin([0, 1])].copy()
df["label"] = df["brugada"].astype(int)

print("Filtered shape (0 vs 1 only):", df.shape)
print("Binary label distribution:")
print(df["label"].value_counts())
print()


# =========================
# 3. ECG LOADER
# =========================
def load_ecg_record(patient_id, base_path=FILES_DIR):
    patient_id = str(patient_id)
    record_path = os.path.join(base_path, patient_id, patient_id)
    record = wfdb.rdrecord(record_path)
    return record.p_signal, record.sig_name, record.fs


# =========================
# 4. FEATURE EXTRACTION
# =========================
def extract_features(signal):
    """
    signal shape: (1200, 12)
    output: 15 fitur per lead x 12 lead = 180 fitur
    """
    feats = []

    for i in range(signal.shape[1]):
        x = signal[:, i]

        feats.extend([
            np.mean(x),
            np.std(x),
            np.min(x),
            np.max(x),
            np.max(x) - np.min(x),
            np.median(x),
            np.percentile(x, 5),
            np.percentile(x, 25),
            np.percentile(x, 75),
            np.percentile(x, 95),
            np.mean(np.abs(x)),
            np.sqrt(np.mean(x ** 2)),
            np.sum(x ** 2),
            skew(x),
            kurtosis(x),
        ])

    return np.array(feats, dtype=float)


# =========================
# 5. BUILD DATASET
# =========================
print("=" * 60)
print("STEP 2 - LOAD ECG & EXTRACT FEATURES")
print("=" * 60)

X, y, ids = [], [], []
failed_ids = []

first_lead_names = None
first_fs = None

for _, row in df.iterrows():
    pid = row["patient_id"]
    label = row["label"]

    try:
        signal, lead_names, fs = load_ecg_record(pid)
        feats = extract_features(signal)

        X.append(feats)
        y.append(label)
        ids.append(pid)

        if first_lead_names is None:
            first_lead_names = lead_names
            first_fs = fs

    except Exception as e:
        failed_ids.append((pid, str(e)))

X = np.array(X)
y = np.array(y)
ids = np.array(ids)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Failed records:", len(failed_ids))
if first_lead_names is not None:
    print("Lead names:", first_lead_names)
    print("Sampling frequency:", first_fs)
print()

if len(failed_ids) > 0:
    print("Sample failed IDs:", failed_ids[:5])
    print()


# =========================
# 6. TRAIN / TEST SPLIT
# =========================
print("=" * 60)
print("STEP 3 - TRAIN/TEST SPLIT")
print("=" * 60)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape :", X_test.shape, y_test.shape)
print()


# =========================
# 7. THRESHOLD TUNING HELPER
# =========================
def evaluate_thresholds(y_true, prob, model_name="MODEL"):
    print("=" * 60)
    print(f"THRESHOLD TUNING - {model_name}")
    print("=" * 60)

    thresholds = np.arange(0.10, 0.91, 0.05)

    best_threshold = None
    best_f1 = -1
    best_bacc = -1
    best_pred = None

    for t in thresholds:
        pred_t = (prob >= t).astype(int)
        f1 = f1_score(y_true, pred_t, zero_division=0)
        bacc = balanced_accuracy_score(y_true, pred_t)

        print(f"Threshold={t:.2f} | F1={f1:.4f} | Balanced Acc={bacc:.4f}")

        if f1 > best_f1:
            best_f1 = f1
            best_bacc = bacc
            best_threshold = t
            best_pred = pred_t

    print()
    print(f"[{model_name}] Best Threshold: {best_threshold:.2f}")
    print(f"[{model_name}] Best F1 Score: {best_f1:.4f}")
    print(f"[{model_name}] Best Balanced Accuracy: {best_bacc:.4f}")
    print(f"[{model_name}] ROC-AUC: {roc_auc_score(y_true, prob):.4f}")
    print()
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, best_pred))
    print()
    print("Classification Report:")
    print(classification_report(y_true, best_pred, zero_division=0))
    print()

    return {
        "best_threshold": best_threshold,
        "best_f1": best_f1,
        "best_bacc": best_bacc,
        "roc_auc": roc_auc_score(y_true, prob),
        "best_pred": best_pred
    }


# =========================
# 8. RANDOM FOREST
# =========================
print("=" * 60)
print("STEP 4 - RANDOM FOREST")
print("=" * 60)

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print("RF probability stats:")
print("Min :", rf_prob.min())
print("Max :", rf_prob.max())
print("Mean:", rf_prob.mean())
print("Top 10 probs:", np.sort(rf_prob)[-10:])
print()

rf_results = evaluate_thresholds(y_test, rf_prob, model_name="RandomForest")


# =========================
# 9. LOGISTIC REGRESSION
# =========================
print("=" * 60)
print("STEP 5 - LOGISTIC REGRESSION")
print("=" * 60)

lr_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    )
)

lr_model.fit(X_train, y_train)
lr_prob = lr_model.predict_proba(X_test)[:, 1]

print("LR probability stats:")
print("Min :", lr_prob.min())
print("Max :", lr_prob.max())
print("Mean:", lr_prob.mean())
print("Top 10 probs:", np.sort(lr_prob)[-10:])
print()

lr_results = evaluate_thresholds(y_test, lr_prob, model_name="LogisticRegression")


# =========================
# 10. FINAL SUMMARY
# =========================
print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)

summary_df = pd.DataFrame([
    {
        "model": "RandomForest",
        "roc_auc": rf_results["roc_auc"],
        "best_threshold": rf_results["best_threshold"],
        "best_f1": rf_results["best_f1"],
        "best_balanced_acc": rf_results["best_bacc"],
    },
    {
        "model": "LogisticRegression",
        "roc_auc": lr_results["roc_auc"],
        "best_threshold": lr_results["best_threshold"],
        "best_f1": lr_results["best_f1"],
        "best_balanced_acc": lr_results["best_bacc"],
    }
])

print(summary_df.sort_values(by="best_f1", ascending=False))
print()

best_model_name = summary_df.sort_values(by="best_f1", ascending=False).iloc[0]["model"]
print("Best current baseline model:", best_model_name)

<>:28: SyntaxWarning: invalid escape sequence '\K'
<>:28: SyntaxWarning: invalid escape sequence '\K'
C:\Users\Owner\AppData\Local\Temp\ipykernel_21716\3820802129.py:28: SyntaxWarning: invalid escape sequence '\K'
  METADATA_PATH = "C:\Kuliah\Lomba\IDSC\Misc\metadata.csv"


STEP 1 - LOAD METADATA
Original shape: (363, 4)
Original brugada distribution:
brugada
0    287
1     69
2      7
Name: count, dtype: int64

Filtered shape (0 vs 1 only): (356, 5)
Binary label distribution:
label
0    287
1     69
Name: count, dtype: int64

STEP 2 - LOAD ECG & EXTRACT FEATURES
X shape: (356, 180)
y shape: (356,)
Failed records: 0
Lead names: ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
Sampling frequency: 100

STEP 3 - TRAIN/TEST SPLIT
Train shape: (284, 180) (284,)
Test shape : (72, 180) (72,)

STEP 4 - RANDOM FOREST
RF probability stats:
Min : 0.023333333333333334
Max : 0.5166666666666667
Mean: 0.15458333333333335
Top 10 probs: [0.27       0.27666667 0.28333333 0.29       0.3        0.34
 0.34333333 0.35333333 0.41       0.51666667]

THRESHOLD TUNING - RandomForest
Threshold=0.10 | F1=0.3810 | Balanced Acc=0.6096
Threshold=0.15 | F1=0.4255 | Balanced Acc=0.6589
Threshold=0.20 | F1=0.3529 | Balanced Acc=0.5936
Threshold=0.25 | F1=0.4286 

In [27]:
import os
import json
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import wfdb
from scipy.signal import welch
from scipy.stats import kurtosis, skew
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# Optional packages
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    from imblearn.over_sampling import RandomOverSampler
    from imblearn.pipeline import Pipeline as ImbPipeline
    HAS_IMBLEARN = True
except Exception:
    HAS_IMBLEARN = False

try:
    import pywt
    HAS_PYWT = True
except Exception:
    HAS_PYWT = False


@dataclass
class Config:
    metadata_path: str = r"C:\Kuliah\Lomba\IDSC\Misc\metadata.csv"
    files_dir: str = r"C:\Kuliah\Lomba\IDSC\Misc\files"
    output_dir: str = "outputs_light"

    random_state: int = 42
    holdout_size: float = 0.2
    n_splits_cv: int = 5

    use_binary_only: bool = True
    include_wavelet_features: bool = False   # False dulu biar enteng
    run_v123_experiment: bool = True

    run_logistic: bool = True
    run_random_forest: bool = True
    run_xgboost: bool = True
    run_resampling_models: bool = False      # matiin dulu biar gak berat
    run_calibration: bool = True

    top_k_features_options: Tuple[int, ...] = (30, 60, 90)
    threshold_grid: Tuple[float, ...] = tuple(np.round(np.arange(0.10, 0.91, 0.02), 2))

    optimize_metric: str = "f1"  # choices: f1, balanced_accuracy, constrained_f1
    min_recall_constraint: float = 0.50

    n_iter_search: int = 12      # jauh lebih ringan dari grid brutal
    n_jobs_search: int = -1      # kalau laptop kepanasan, ganti 1 atau 2

    save_oof_predictions: bool = True


CFG = Config()


def ensure_output_dirs(cfg: Config) -> Dict[str, Path]:
    base = Path(cfg.output_dir)
    paths = {
        "base": base,
        "models": base / "models",
        "metrics": base / "metrics",
        "predictions": base / "predictions",
    }
    for p in paths.values():
        p.mkdir(parents=True, exist_ok=True)
    return paths


def save_json(obj: Any, path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def save_dataframe(df: pd.DataFrame, path: Path) -> None:
    df.to_csv(path, index=False)


def print_environment_notes() -> None:
    print("=" * 80)
    print("ENVIRONMENT NOTES")
    print("=" * 80)
    print(f"XGBoost available   : {HAS_XGB}")
    print(f"imblearn available  : {HAS_IMBLEARN}")
    print(f"PyWavelets available: {HAS_PYWT}")
    print()
    if not HAS_XGB:
        print("WARNING: xgboost not installed. XGBoost will be skipped.")
    if not HAS_IMBLEARN:
        print("WARNING: imbalanced-learn not installed. Resampling models skipped.")
    if not HAS_PYWT:
        print("WARNING: PyWavelets not installed. Wavelet features skipped.")
    print()


def load_metadata(cfg: Config) -> pd.DataFrame:
    df = pd.read_csv(cfg.metadata_path)
    required = {"patient_id", "brugada"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    if cfg.use_binary_only:
        df = df[df["brugada"].isin([0, 1])].copy()
        df["label"] = df["brugada"].astype(int)
    else:
        raise ValueError("Script ini diset untuk binary 0 vs 1 dulu.")

    df = df.drop_duplicates(subset=["patient_id"]).reset_index(drop=True)
    return df


def load_ecg_record(patient_id: Any, files_dir: str) -> Tuple[np.ndarray, List[str], float]:
    pid = str(patient_id)
    record_path = os.path.join(files_dir, pid, pid)
    record = wfdb.rdrecord(record_path)
    return record.p_signal.astype(np.float32), list(record.sig_name), float(record.fs)


def safe_entropy(values: np.ndarray, bins: int = 32) -> float:
    hist, _ = np.histogram(values, bins=bins, density=True)
    hist = hist[hist > 0]
    if len(hist) == 0:
        return 0.0
    return float(-np.sum(hist * np.log(hist + 1e-12)))


def base_stats_1d(x: np.ndarray) -> List[float]:
    x = np.asarray(x)
    dx = np.diff(x)
    zero_cross = float(np.mean(np.diff(np.signbit(x)).astype(int))) if len(x) > 1 else 0.0

    return [
        float(np.mean(x)),
        float(np.std(x)),
        float(np.min(x)),
        float(np.max(x)),
        float(np.max(x) - np.min(x)),
        float(np.median(x)),
        float(np.percentile(x, 5)),
        float(np.percentile(x, 25)),
        float(np.percentile(x, 75)),
        float(np.percentile(x, 95)),
        float(np.mean(np.abs(x))),
        float(np.sqrt(np.mean(x ** 2))),
        float(np.sum(x ** 2)),
        float(skew(x)),
        float(kurtosis(x)),
        float(np.mean(dx)) if len(dx) else 0.0,
        float(np.std(dx)) if len(dx) else 0.0,
        float(np.max(np.abs(dx))) if len(dx) else 0.0,
        zero_cross,
        float(safe_entropy(x)),
    ]


def spectral_features_1d(x: np.ndarray, fs: float) -> List[float]:
    freqs, psd = welch(x, fs=fs, nperseg=min(len(x), 256))
    if len(freqs) == 0 or len(psd) == 0:
        return [0.0] * 6

    psd = np.maximum(psd, 1e-12)
    total_power = float(np.trapz(psd, freqs))

    def bandpower(low: float, high: float) -> float:
        mask = (freqs >= low) & (freqs < high)
        if not np.any(mask):
            return 0.0
        return float(np.trapz(psd[mask], freqs[mask]))

    power_0_5 = bandpower(0.0, 5.0)
    power_5_15 = bandpower(5.0, 15.0)
    power_15_40 = bandpower(15.0, 40.0)
    peak_freq = float(freqs[np.argmax(psd)])
    p = psd / psd.sum()
    spec_entropy = float(-np.sum(p * np.log(p + 1e-12)))

    return [
        total_power,
        power_0_5,
        power_5_15,
        power_15_40,
        peak_freq,
        spec_entropy,
    ]


def wavelet_features_1d(x: np.ndarray) -> List[float]:
    if not HAS_PYWT:
        return []
    coeffs = pywt.wavedec(x, "db4", level=3)
    feats: List[float] = []
    for c in coeffs:
        c = np.asarray(c)
        feats.extend([
            float(np.mean(c)),
            float(np.std(c)),
            float(np.sum(c ** 2)),
            float(safe_entropy(c)),
        ])
    return feats


def extract_features(signal: np.ndarray, fs: float, include_wavelet: bool = False) -> np.ndarray:
    feats: List[float] = []
    for i in range(signal.shape[1]):
        x = signal[:, i]
        feats.extend(base_stats_1d(x))
        feats.extend(spectral_features_1d(x, fs))
        if include_wavelet:
            feats.extend(wavelet_features_1d(x))
    return np.asarray(feats, dtype=np.float32)


def lead_indices_from_names(lead_names: List[str], target_leads: List[str]) -> List[int]:
    mapping = {name: idx for idx, name in enumerate(lead_names)}
    return [mapping[name] for name in target_leads if name in mapping]


def build_feature_table(
    df: pd.DataFrame,
    cfg: Config,
    use_v123_only: bool = False
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, List[str], float, List[Tuple[Any, str]]]:
    X, y, ids = [], [], []
    failed: List[Tuple[Any, str]] = []
    lead_names: Optional[List[str]] = None
    fs_final: Optional[float] = None

    for _, row in df.iterrows():
        pid = row["patient_id"]
        label = int(row["label"])
        try:
            signal, current_leads, fs = load_ecg_record(pid, cfg.files_dir)

            if lead_names is None:
                lead_names = current_leads
                fs_final = fs

            if use_v123_only:
                idx = lead_indices_from_names(current_leads, ["V1", "V2", "V3"])
                if len(idx) != 3:
                    raise ValueError(f"V1-V3 not found for patient {pid}")
                signal = signal[:, idx]

            feats = extract_features(signal, fs, include_wavelet=cfg.include_wavelet_features)
            X.append(feats)
            y.append(label)
            ids.append(pid)

        except Exception as e:
            failed.append((pid, str(e)))

    if not X:
        raise RuntimeError("No ECG sample loaded successfully.")

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.int32),
        np.asarray(ids),
        lead_names or [],
        float(fs_final if fs_final is not None else 100.0),
        failed,
    )


def build_model_search_spaces(cfg: Config) -> Dict[str, Tuple[Any, Dict[str, List[Any]]]]:
    models: Dict[str, Tuple[Any, Dict[str, List[Any]]]] = {}

    if cfg.run_logistic:
        log_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("select", SelectKBest(score_func=f_classif)),
            ("clf", LogisticRegression(max_iter=4000, random_state=cfg.random_state)),
        ])
        log_dist = {
            "select__k": list(cfg.top_k_features_options),
            "clf__C": [0.03, 0.1, 0.3, 1.0, 3.0, 10.0],
            "clf__class_weight": ["balanced", {0: 1, 1: 2}, {0: 1, 1: 3}],
            "clf__solver": ["lbfgs"],
            "clf__penalty": ["l2"],
        }
        models["logistic"] = (log_pipe, log_dist)

    if cfg.run_random_forest:
        rf_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("select", SelectKBest(score_func=f_classif)),
            ("clf", RandomForestClassifier(random_state=cfg.random_state, n_jobs=-1)),
        ])
        rf_dist = {
            "select__k": list(cfg.top_k_features_options),
            "clf__n_estimators": [200, 300, 500],
            "clf__max_depth": [4, 6, 10, None],
            "clf__min_samples_split": [2, 5, 10],
            "clf__min_samples_leaf": [1, 2, 4],
            "clf__max_features": ["sqrt", 0.5],
            "clf__class_weight": ["balanced", {0: 1, 1: 2}, {0: 1, 1: 3}],
        }
        models["random_forest"] = (rf_pipe, rf_dist)

    if cfg.run_xgboost and HAS_XGB:
        xgb_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("select", SelectKBest(score_func=f_classif)),
            ("clf", XGBClassifier(
                random_state=cfg.random_state,
                eval_metric="logloss",
                tree_method="hist",
                n_jobs=max(1, os.cpu_count() // 2),
            )),
        ])
        xgb_dist = {
            "select__k": list(cfg.top_k_features_options),
            "clf__n_estimators": [150, 250, 400],
            "clf__learning_rate": [0.03, 0.05, 0.1],
            "clf__max_depth": [2, 3, 4],
            "clf__min_child_weight": [1, 3, 5],
            "clf__subsample": [0.7, 0.85, 1.0],
            "clf__colsample_bytree": [0.6, 0.8, 1.0],
            "clf__reg_alpha": [0.0, 0.1, 1.0],
            "clf__reg_lambda": [1.0, 2.0, 5.0],
            "clf__scale_pos_weight": [1.0, 2.0, 3.0],
        }
        models["xgboost"] = (xgb_pipe, xgb_dist)

    if cfg.run_resampling_models and HAS_IMBLEARN:
        ros_log_pipe = ImbPipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("select", SelectKBest(score_func=f_classif)),
            ("ros", RandomOverSampler(random_state=cfg.random_state)),
            ("clf", LogisticRegression(max_iter=4000, random_state=cfg.random_state)),
        ])
        ros_log_dist = {
            "select__k": list(cfg.top_k_features_options),
            "ros__sampling_strategy": [0.5, 0.75, 1.0],
            "clf__C": [0.1, 1.0, 3.0],
            "clf__class_weight": [None, "balanced"],
            "clf__solver": ["lbfgs"],
        }
        models["ros_logistic"] = (ros_log_pipe, ros_log_dist)

    return models


def optimize_threshold(y_true: np.ndarray, prob: np.ndarray, cfg: Config) -> Dict[str, Any]:
    best_score = -np.inf
    best_threshold = 0.5
    best_metrics: Dict[str, Any] = {}

    for t in cfg.threshold_grid:
        pred = (prob >= t).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        bacc = balanced_accuracy_score(y_true, pred)
        prec = precision_score(y_true, pred, zero_division=0)
        rec = recall_score(y_true, pred, zero_division=0)

        if cfg.optimize_metric == "f1":
            score = f1
        elif cfg.optimize_metric == "balanced_accuracy":
            score = bacc
        elif cfg.optimize_metric == "constrained_f1":
            score = f1 if rec >= cfg.min_recall_constraint else -1e9 + rec
        else:
            raise ValueError(f"Unknown optimize_metric: {cfg.optimize_metric}")

        if score > best_score:
            best_score = score
            best_threshold = float(t)
            best_metrics = {
                "threshold": float(t),
                "f1": float(f1),
                "balanced_accuracy": float(bacc),
                "precision": float(prec),
                "recall": float(rec),
                "confusion_matrix": confusion_matrix(y_true, pred).tolist(),
            }

    best_metrics["roc_auc"] = float(roc_auc_score(y_true, prob))
    best_metrics["pr_auc"] = float(average_precision_score(y_true, prob))
    return best_metrics


def oof_predictions_from_best_estimator(best_estimator: Any, X_train: np.ndarray, y_train: np.ndarray, cfg: Config) -> np.ndarray:
    skf = StratifiedKFold(n_splits=cfg.n_splits_cv, shuffle=True, random_state=cfg.random_state)
    oof_prob = np.zeros(len(y_train), dtype=float)

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), start=1):
        model = clone(best_estimator)
        model.fit(X_train[tr_idx], y_train[tr_idx])
        oof_prob[val_idx] = model.predict_proba(X_train[val_idx])[:, 1]
        print(f"    OOF fold {fold}/{cfg.n_splits_cv} done")

    return oof_prob


def run_random_search(name: str, estimator: Any, param_dist: Dict[str, List[Any]], X_train: np.ndarray, y_train: np.ndarray, cfg: Config) -> RandomizedSearchCV:
    cv = StratifiedKFold(n_splits=cfg.n_splits_cv, shuffle=True, random_state=cfg.random_state)

    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_dist,
        n_iter=cfg.n_iter_search,
        scoring={
            "roc_auc": "roc_auc",
            "f1": "f1",
            "balanced_accuracy": "balanced_accuracy",
            "average_precision": "average_precision",
        },
        refit="f1",
        cv=cv,
        n_jobs=cfg.n_jobs_search,
        verbose=1,
        random_state=cfg.random_state,
        return_train_score=False,
    )

    print(f"\n===== Running RandomizedSearch for {name} =====")
    search.fit(X_train, y_train)
    print(f"Best params for {name}: {search.best_params_}")
    print(f"Best CV F1 for {name}: {search.best_score_:.4f}")
    return search


def maybe_calibrate_model(best_estimator: Any, X_train: np.ndarray, y_train: np.ndarray, cfg: Config) -> Any:
    if not cfg.run_calibration:
        return best_estimator
    cv = StratifiedKFold(n_splits=cfg.n_splits_cv, shuffle=True, random_state=cfg.random_state)
    calibrated = CalibratedClassifierCV(best_estimator, method="sigmoid", cv=cv)
    calibrated.fit(X_train, y_train)
    return calibrated


def evaluate_on_holdout(model: Any, X_test: np.ndarray, y_test: np.ndarray, threshold: float) -> Dict[str, Any]:
    prob = model.predict_proba(X_test)[:, 1]
    pred = (prob >= threshold).astype(int)

    return {
        "threshold": float(threshold),
        "roc_auc": float(roc_auc_score(y_test, prob)),
        "pr_auc": float(average_precision_score(y_test, prob)),
        "f1": float(f1_score(y_test, pred, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_test, pred)),
        "precision": float(precision_score(y_test, pred, zero_division=0)),
        "recall": float(recall_score(y_test, pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_test, pred).tolist(),
        "classification_report": classification_report(y_test, pred, zero_division=0, output_dict=True),
        "probabilities": prob.tolist(),
        "predictions": pred.tolist(),
    }


def run_experiment_suite(
    X: np.ndarray,
    y: np.ndarray,
    ids: np.ndarray,
    lead_mode: str,
    cfg: Config,
    output_paths: Dict[str, Path],
) -> pd.DataFrame:
    print("\n" + "=" * 80)
    print(f"EXPERIMENT SUITE START: {lead_mode}")
    print("=" * 80)

    X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
        X, y, ids,
        test_size=cfg.holdout_size,
        stratify=y,
        random_state=cfg.random_state,
    )

    print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")
    print(f"Train positives: {int(y_train.sum())} / {len(y_train)}")
    print(f"Test positives : {int(y_test.sum())} / {len(y_test)}")

    model_spaces = build_model_search_spaces(cfg)
    results_rows: List[Dict[str, Any]] = []
    holdout_prediction_records: List[Dict[str, Any]] = []

    for model_name, (estimator, param_dist) in model_spaces.items():
        search = run_random_search(model_name, estimator, param_dist, X_train, y_train, cfg)

        cv_results = pd.DataFrame(search.cv_results_).sort_values(by="rank_test_f1")
        save_dataframe(cv_results, output_paths["metrics"] / f"cv_results_{lead_mode}_{model_name}.csv")

        print(f"Building OOF predictions for {model_name}...")
        oof_prob = oof_predictions_from_best_estimator(search.best_estimator_, X_train, y_train, cfg)
        threshold_metrics = optimize_threshold(y_train, oof_prob, cfg)
        best_threshold = threshold_metrics["threshold"]

        model_for_holdout = maybe_calibrate_model(search.best_estimator_, X_train, y_train, cfg)
        holdout_metrics = evaluate_on_holdout(model_for_holdout, X_test, y_test, best_threshold)

        results_rows.append({
            "lead_mode": lead_mode,
            "model": model_name,
            "cv_best_f1": float(search.best_score_),
            "oof_threshold": best_threshold,
            "oof_f1": threshold_metrics["f1"],
            "oof_balanced_accuracy": threshold_metrics["balanced_accuracy"],
            "oof_precision": threshold_metrics["precision"],
            "oof_recall": threshold_metrics["recall"],
            "oof_roc_auc": threshold_metrics["roc_auc"],
            "oof_pr_auc": threshold_metrics["pr_auc"],
            "holdout_f1": holdout_metrics["f1"],
            "holdout_balanced_accuracy": holdout_metrics["balanced_accuracy"],
            "holdout_precision": holdout_metrics["precision"],
            "holdout_recall": holdout_metrics["recall"],
            "holdout_roc_auc": holdout_metrics["roc_auc"],
            "holdout_pr_auc": holdout_metrics["pr_auc"],
            "best_params": json.dumps(search.best_params_),
        })

        detail = {
            "lead_mode": lead_mode,
            "model": model_name,
            "best_params": search.best_params_,
            "oof_threshold_metrics": threshold_metrics,
            "holdout_metrics": holdout_metrics,
        }
        save_json(detail, output_paths["metrics"] / f"details_{lead_mode}_{model_name}.json")

        if cfg.save_oof_predictions:
            pred_df = pd.DataFrame({
                "patient_id": ids_train,
                "y_true": y_train,
                "oof_prob": oof_prob,
            })
            save_dataframe(pred_df, output_paths["predictions"] / f"oof_predictions_{lead_mode}_{model_name}.csv")

        holdout_prediction_records.append({
            "lead_mode": lead_mode,
            "model": model_name,
            "ids_test": ids_test.tolist(),
            "y_test": y_test.tolist(),
            "prob": holdout_metrics["probabilities"],
            "pred": holdout_metrics["predictions"],
            "threshold": best_threshold,
        })

    save_json(holdout_prediction_records, output_paths["predictions"] / f"holdout_predictions_{lead_mode}.json")

    results_df = pd.DataFrame(results_rows).sort_values(
        by=["holdout_f1", "holdout_balanced_accuracy", "holdout_roc_auc"],
        ascending=False,
    )

    save_dataframe(results_df, output_paths["metrics"] / f"summary_{lead_mode}.csv")

    print("\nTop results:")
    print(results_df.head(10).to_string(index=False))
    return results_df


def main() -> None:
    print_environment_notes()
    output_paths = ensure_output_dirs(CFG)
    save_json(asdict(CFG), output_paths["base"] / "config.json")

    df = load_metadata(CFG)

    print("=" * 80)
    print("DATASET OVERVIEW")
    print("=" * 80)
    print(f"Metadata shape after filtering: {df.shape}")
    print(df["label"].value_counts().sort_index())
    print()

    X_all, y_all, ids_all, lead_names, fs, failed_all = build_feature_table(df, CFG, use_v123_only=False)
    print("Built 12-lead feature table")
    print(f"X_all shape: {X_all.shape}")
    print(f"y_all shape: {y_all.shape}")
    print(f"Lead names : {lead_names}")
    print(f"Sampling fs: {fs}")
    print(f"Failed records: {len(failed_all)}")
    if failed_all:
        print("Sample failures:", failed_all[:5])
    print()

    all_results = []

    results_12 = run_experiment_suite(X_all, y_all, ids_all, "all_12_leads", CFG, output_paths)
    all_results.append(results_12)

    if CFG.run_v123_experiment:
        X_v, y_v, ids_v, _, _, failed_v = build_feature_table(df, CFG, use_v123_only=True)
        print("\nBuilt V1-V3 feature table")
        print(f"X_v123 shape: {X_v.shape}")
        print(f"y_v123 shape: {y_v.shape}")
        print(f"Failed records V1-V3: {len(failed_v)}")

        results_v = run_experiment_suite(X_v, y_v, ids_v, "v1_v2_v3_only", CFG, output_paths)
        all_results.append(results_v)

    final_summary = pd.concat(all_results, axis=0, ignore_index=True).sort_values(
        by=["holdout_f1", "holdout_balanced_accuracy", "holdout_roc_auc"],
        ascending=False,
    )

    save_dataframe(final_summary, output_paths["base"] / "final_summary_all_experiments.csv")

    print("\n" + "=" * 80)
    print("FINAL LEADERBOARD")
    print("=" * 80)
    print(final_summary.to_string(index=False))
    print()

    best_row = final_summary.iloc[0].to_dict()
    save_json(best_row, output_paths["base"] / "best_model_summary.json")

    print("Best experiment:")
    print(json.dumps(best_row, indent=2))
    print("\nAll results saved under:", output_paths["base"].resolve())


if __name__ == "__main__":
    main()

ENVIRONMENT NOTES
XGBoost available   : True
imblearn available  : True
PyWavelets available: True


DATASET OVERVIEW
Metadata shape after filtering: (356, 5)
label
0    287
1     69
Name: count, dtype: int64

Built 12-lead feature table
X_all shape: (356, 312)
y_all shape: (356,)
Lead names : ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
Sampling fs: 100.0
Failed records: 0


EXPERIMENT SUITE START: all_12_leads
Train size: (284, 312), Test size: (72, 312)
Train positives: 55 / 284
Test positives : 14 / 72

===== Running RandomizedSearch for logistic =====
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best params for logistic: {'select__k': 60, 'clf__solver': 'lbfgs', 'clf__penalty': 'l2', 'clf__class_weight': 'balanced', 'clf__C': 0.3}
Best CV F1 for logistic: 0.5463
Building OOF predictions for logistic...
    OOF fold 1/5 done
    OOF fold 2/5 done
    OOF fold 3/5 done
    OOF fold 4/5 done
    OOF fold 5/5 done

===== Running Randomize

In [28]:
import os
import json
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import wfdb
from scipy.signal import welch
from scipy.stats import kurtosis, skew
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# =========================
# Optional packages
# =========================
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    from imblearn.over_sampling import RandomOverSampler
    from imblearn.pipeline import Pipeline as ImbPipeline
    HAS_IMBLEARN = True
except Exception:
    HAS_IMBLEARN = False

try:
    import pywt
    HAS_PYWT = True
except Exception:
    HAS_PYWT = False


# =========================
# Config
# =========================
@dataclass
class Config:
    metadata_path: str = r"C:\Kuliah\Lomba\IDSC\Misc\metadata.csv"
    files_dir: str = r"C:\Kuliah\Lomba\IDSC\Misc\files"
    output_dir: str = "outputs_round2_v123"

    random_state: int = 42
    holdout_size: float = 0.2
    n_splits_cv: int = 5

    use_binary_only: bool = True
    use_v123_only: bool = True

    include_wavelet_features: bool = True   # round 2: ON
    save_oof_predictions: bool = True
    run_calibration: bool = True

    # Fokus model
    run_logistic: bool = True
    run_random_forest: bool = True
    run_xgboost: bool = True
    run_ros_logistic: bool = True

    # Search size: masih cukup ringan
    n_iter_search: int = 10
    n_jobs_search: int = -1

    # K feature
    top_k_features_options: Tuple[int, ...] = (30, 45, 60, 75, 90)

    # Threshold tuning
    threshold_grid: Tuple[float, ...] = tuple(np.round(np.arange(0.10, 0.91, 0.01), 2))
    optimize_metric: str = "constrained_f1"   # choices: f1, balanced_accuracy, constrained_f1
    min_recall_constraint: float = 0.50       # bisa lu naikin jadi 0.60 kalau mau lebih agresif

    # Fallback ranking if many satisfy recall constraint
    secondary_sort_metric: str = "balanced_accuracy"


CFG = Config()


# =========================
# Utils
# =========================
def ensure_output_dirs(cfg: Config) -> Dict[str, Path]:
    base = Path(cfg.output_dir)
    paths = {
        "base": base,
        "models": base / "models",
        "metrics": base / "metrics",
        "predictions": base / "predictions",
    }
    for p in paths.values():
        p.mkdir(parents=True, exist_ok=True)
    return paths


def save_json(obj: Any, path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def save_dataframe(df: pd.DataFrame, path: Path) -> None:
    df.to_csv(path, index=False)


def print_environment_notes() -> None:
    print("=" * 80)
    print("ENVIRONMENT NOTES")
    print("=" * 80)
    print(f"XGBoost available   : {HAS_XGB}")
    print(f"imblearn available  : {HAS_IMBLEARN}")
    print(f"PyWavelets available: {HAS_PYWT}")
    print()
    if not HAS_XGB:
        print("WARNING: xgboost not installed. XGBoost will be skipped.")
    if not HAS_IMBLEARN:
        print("WARNING: imbalanced-learn not installed. ROS logistic will be skipped.")
    if not HAS_PYWT:
        print("WARNING: PyWavelets not installed. Wavelet features will be skipped.")
    print()


# =========================
# Data loading
# =========================
def load_metadata(cfg: Config) -> pd.DataFrame:
    df = pd.read_csv(cfg.metadata_path)
    required = {"patient_id", "brugada"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    if cfg.use_binary_only:
        df = df[df["brugada"].isin([0, 1])].copy()
        df["label"] = df["brugada"].astype(int)
    else:
        raise ValueError("Script ini fokus binary 0 vs 1.")

    df = df.drop_duplicates(subset=["patient_id"]).reset_index(drop=True)
    return df


def load_ecg_record(patient_id: Any, files_dir: str) -> Tuple[np.ndarray, List[str], float]:
    pid = str(patient_id)
    record_path = os.path.join(files_dir, pid, pid)
    record = wfdb.rdrecord(record_path)
    return record.p_signal.astype(np.float32), list(record.sig_name), float(record.fs)


def lead_indices_from_names(lead_names: List[str], target_leads: List[str]) -> List[int]:
    mapping = {name: idx for idx, name in enumerate(lead_names)}
    return [mapping[name] for name in target_leads if name in mapping]


# =========================
# Feature extraction
# =========================
def safe_entropy(values: np.ndarray, bins: int = 32) -> float:
    hist, _ = np.histogram(values, bins=bins, density=True)
    hist = hist[hist > 0]
    if len(hist) == 0:
        return 0.0
    return float(-np.sum(hist * np.log(hist + 1e-12)))


def base_stats_1d(x: np.ndarray) -> List[float]:
    x = np.asarray(x)
    dx = np.diff(x)
    zero_cross = float(np.mean(np.diff(np.signbit(x)).astype(int))) if len(x) > 1 else 0.0

    return [
        float(np.mean(x)),
        float(np.std(x)),
        float(np.min(x)),
        float(np.max(x)),
        float(np.max(x) - np.min(x)),
        float(np.median(x)),
        float(np.percentile(x, 5)),
        float(np.percentile(x, 25)),
        float(np.percentile(x, 75)),
        float(np.percentile(x, 95)),
        float(np.mean(np.abs(x))),
        float(np.sqrt(np.mean(x ** 2))),
        float(np.sum(x ** 2)),
        float(skew(x)),
        float(kurtosis(x)),
        float(np.mean(dx)) if len(dx) else 0.0,
        float(np.std(dx)) if len(dx) else 0.0,
        float(np.max(np.abs(dx))) if len(dx) else 0.0,
        zero_cross,
        float(safe_entropy(x)),
    ]


def spectral_features_1d(x: np.ndarray, fs: float) -> List[float]:
    freqs, psd = welch(x, fs=fs, nperseg=min(len(x), 256))
    if len(freqs) == 0 or len(psd) == 0:
        return [0.0] * 6

    psd = np.maximum(psd, 1e-12)
    total_power = float(np.trapz(psd, freqs))

    def bandpower(low: float, high: float) -> float:
        mask = (freqs >= low) & (freqs < high)
        if not np.any(mask):
            return 0.0
        return float(np.trapz(psd[mask], freqs[mask]))

    power_0_5 = bandpower(0.0, 5.0)
    power_5_15 = bandpower(5.0, 15.0)
    power_15_40 = bandpower(15.0, 40.0)
    peak_freq = float(freqs[np.argmax(psd)])
    p = psd / psd.sum()
    spec_entropy = float(-np.sum(p * np.log(p + 1e-12)))

    return [
        total_power,
        power_0_5,
        power_5_15,
        power_15_40,
        peak_freq,
        spec_entropy,
    ]


def wavelet_features_1d(x: np.ndarray) -> List[float]:
    if not HAS_PYWT:
        return []
    coeffs = pywt.wavedec(x, "db4", level=3)
    feats: List[float] = []
    for c in coeffs:
        c = np.asarray(c)
        feats.extend([
            float(np.mean(c)),
            float(np.std(c)),
            float(np.sum(c ** 2)),
            float(safe_entropy(c)),
        ])
    return feats


def extract_features(signal: np.ndarray, fs: float, include_wavelet: bool = True) -> np.ndarray:
    feats: List[float] = []
    for i in range(signal.shape[1]):
        x = signal[:, i]
        feats.extend(base_stats_1d(x))
        feats.extend(spectral_features_1d(x, fs))
        if include_wavelet:
            feats.extend(wavelet_features_1d(x))
    return np.asarray(feats, dtype=np.float32)


def build_feature_table(
    df: pd.DataFrame,
    cfg: Config,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, List[str], float, List[Tuple[Any, str]]]:
    X, y, ids = [], [], []
    failed: List[Tuple[Any, str]] = []
    lead_names: Optional[List[str]] = None
    fs_final: Optional[float] = None

    for _, row in df.iterrows():
        pid = row["patient_id"]
        label = int(row["label"])
        try:
            signal, current_leads, fs = load_ecg_record(pid, cfg.files_dir)

            if lead_names is None:
                lead_names = current_leads
                fs_final = fs

            if cfg.use_v123_only:
                idx = lead_indices_from_names(current_leads, ["V1", "V2", "V3"])
                if len(idx) != 3:
                    raise ValueError(f"V1-V3 not found for patient {pid}")
                signal = signal[:, idx]

            feats = extract_features(signal, fs, include_wavelet=cfg.include_wavelet_features)
            X.append(feats)
            y.append(label)
            ids.append(pid)

        except Exception as e:
            failed.append((pid, str(e)))

    if not X:
        raise RuntimeError("No ECG sample loaded successfully.")

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.int32),
        np.asarray(ids),
        lead_names or [],
        float(fs_final if fs_final is not None else 100.0),
        failed,
    )


# =========================
# Model zoo
# =========================
def build_model_search_spaces(cfg: Config) -> Dict[str, Tuple[Any, Dict[str, List[Any]]]]:
    models: Dict[str, Tuple[Any, Dict[str, List[Any]]]] = {}

    if cfg.run_logistic:
        log_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("select", SelectKBest(score_func=f_classif)),
            ("clf", LogisticRegression(max_iter=5000, random_state=cfg.random_state)),
        ])
        log_dist = {
            "select__k": list(cfg.top_k_features_options),
            "clf__C": [0.03, 0.1, 0.3, 1.0, 3.0],
            "clf__class_weight": ["balanced", {0: 1, 1: 2}, {0: 1, 1: 3}, {0: 1, 1: 4}],
            "clf__solver": ["lbfgs"],
            "clf__penalty": ["l2"],
        }
        models["logistic"] = (log_pipe, log_dist)

    if cfg.run_random_forest:
        rf_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("select", SelectKBest(score_func=f_classif)),
            ("clf", RandomForestClassifier(random_state=cfg.random_state, n_jobs=-1)),
        ])
        rf_dist = {
            "select__k": list(cfg.top_k_features_options),
            "clf__n_estimators": [200, 300, 500],
            "clf__max_depth": [3, 4, 6, 8],
            "clf__min_samples_split": [2, 5, 10],
            "clf__min_samples_leaf": [1, 2, 4],
            "clf__max_features": ["sqrt", 0.5],
            "clf__class_weight": ["balanced", {0: 1, 1: 2}, {0: 1, 1: 3}, {0: 1, 1: 4}],
        }
        models["random_forest"] = (rf_pipe, rf_dist)

    if cfg.run_xgboost and HAS_XGB:
        xgb_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("select", SelectKBest(score_func=f_classif)),
            ("clf", XGBClassifier(
                random_state=cfg.random_state,
                eval_metric="logloss",
                tree_method="hist",
                n_jobs=max(1, os.cpu_count() // 2),
            )),
        ])
        xgb_dist = {
            "select__k": list(cfg.top_k_features_options),
            "clf__n_estimators": [120, 180, 250, 350],
            "clf__learning_rate": [0.02, 0.03, 0.05, 0.08],
            "clf__max_depth": [2, 3, 4],
            "clf__min_child_weight": [1, 3, 5],
            "clf__subsample": [0.7, 0.85, 1.0],
            "clf__colsample_bytree": [0.6, 0.8, 1.0],
            "clf__reg_alpha": [0.0, 0.1, 0.5, 1.0],
            "clf__reg_lambda": [1.0, 2.0, 5.0],
            "clf__scale_pos_weight": [2.0, 3.0, 4.0, 5.0],
        }
        models["xgboost"] = (xgb_pipe, xgb_dist)

    if cfg.run_ros_logistic and HAS_IMBLEARN:
        ros_log_pipe = ImbPipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("select", SelectKBest(score_func=f_classif)),
            ("ros", RandomOverSampler(random_state=cfg.random_state)),
            ("clf", LogisticRegression(max_iter=5000, random_state=cfg.random_state)),
        ])
        ros_log_dist = {
            "select__k": list(cfg.top_k_features_options),
            "ros__sampling_strategy": [0.5, 0.75, 1.0],
            "clf__C": [0.03, 0.1, 0.3, 1.0, 3.0],
            "clf__class_weight": [None, "balanced", {0: 1, 1: 2}],
            "clf__solver": ["lbfgs"],
            "clf__penalty": ["l2"],
        }
        models["ros_logistic"] = (ros_log_pipe, ros_log_dist)

    return models


# =========================
# Threshold optimization
# =========================
def optimize_threshold(y_true: np.ndarray, prob: np.ndarray, cfg: Config) -> Dict[str, Any]:
    rows = []
    best_score = -np.inf
    best_threshold = 0.5
    best_metrics: Dict[str, Any] = {}

    for t in cfg.threshold_grid:
        pred = (prob >= t).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        bacc = balanced_accuracy_score(y_true, pred)
        prec = precision_score(y_true, pred, zero_division=0)
        rec = recall_score(y_true, pred, zero_division=0)

        if cfg.optimize_metric == "f1":
            score = f1
        elif cfg.optimize_metric == "balanced_accuracy":
            score = bacc
        elif cfg.optimize_metric == "constrained_f1":
            if rec >= cfg.min_recall_constraint:
                if cfg.secondary_sort_metric == "balanced_accuracy":
                    score = f1 + 1e-3 * bacc
                else:
                    score = f1
            else:
                score = -1e6 + rec
        else:
            raise ValueError(f"Unknown optimize_metric: {cfg.optimize_metric}")

        row = {
            "threshold": float(t),
            "f1": float(f1),
            "balanced_accuracy": float(bacc),
            "precision": float(prec),
            "recall": float(rec),
            "score_for_selection": float(score),
        }
        rows.append(row)

        if score > best_score:
            best_score = score
            best_threshold = float(t)
            best_metrics = {
                "threshold": float(t),
                "f1": float(f1),
                "balanced_accuracy": float(bacc),
                "precision": float(prec),
                "recall": float(rec),
                "confusion_matrix": confusion_matrix(y_true, pred).tolist(),
            }

    best_metrics["roc_auc"] = float(roc_auc_score(y_true, prob))
    best_metrics["pr_auc"] = float(average_precision_score(y_true, prob))
    best_metrics["threshold_table"] = rows
    return best_metrics


# =========================
# CV / search / eval
# =========================
def oof_predictions_from_best_estimator(best_estimator: Any, X_train: np.ndarray, y_train: np.ndarray, cfg: Config) -> np.ndarray:
    skf = StratifiedKFold(n_splits=cfg.n_splits_cv, shuffle=True, random_state=cfg.random_state)
    oof_prob = np.zeros(len(y_train), dtype=float)

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), start=1):
        model = clone(best_estimator)
        model.fit(X_train[tr_idx], y_train[tr_idx])
        oof_prob[val_idx] = model.predict_proba(X_train[val_idx])[:, 1]
        print(f"    OOF fold {fold}/{cfg.n_splits_cv} done")

    return oof_prob


def run_random_search(name: str, estimator: Any, param_dist: Dict[str, List[Any]], X_train: np.ndarray, y_train: np.ndarray, cfg: Config) -> RandomizedSearchCV:
    cv = StratifiedKFold(n_splits=cfg.n_splits_cv, shuffle=True, random_state=cfg.random_state)

    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_dist,
        n_iter=cfg.n_iter_search,
        scoring={
            "roc_auc": "roc_auc",
            "f1": "f1",
            "balanced_accuracy": "balanced_accuracy",
            "average_precision": "average_precision",
            "recall": "recall",
        },
        refit="f1",
        cv=cv,
        n_jobs=cfg.n_jobs_search,
        verbose=1,
        random_state=cfg.random_state,
        return_train_score=False,
    )

    print(f"\n===== Running RandomizedSearch for {name} =====")
    search.fit(X_train, y_train)
    print(f"Best params for {name}: {search.best_params_}")
    print(f"Best CV F1 for {name}: {search.best_score_:.4f}")
    return search


def maybe_calibrate_model(best_estimator: Any, X_train: np.ndarray, y_train: np.ndarray, cfg: Config) -> Any:
    if not cfg.run_calibration:
        return best_estimator
    cv = StratifiedKFold(n_splits=cfg.n_splits_cv, shuffle=True, random_state=cfg.random_state)
    calibrated = CalibratedClassifierCV(best_estimator, method="sigmoid", cv=cv)
    calibrated.fit(X_train, y_train)
    return calibrated


def evaluate_on_holdout(model: Any, X_test: np.ndarray, y_test: np.ndarray, threshold: float) -> Dict[str, Any]:
    prob = model.predict_proba(X_test)[:, 1]
    pred = (prob >= threshold).astype(int)

    return {
        "threshold": float(threshold),
        "roc_auc": float(roc_auc_score(y_test, prob)),
        "pr_auc": float(average_precision_score(y_test, prob)),
        "f1": float(f1_score(y_test, pred, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_test, pred)),
        "precision": float(precision_score(y_test, pred, zero_division=0)),
        "recall": float(recall_score(y_test, pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_test, pred).tolist(),
        "classification_report": classification_report(y_test, pred, zero_division=0, output_dict=True),
        "probabilities": prob.tolist(),
        "predictions": pred.tolist(),
    }


# =========================
# Main experiment
# =========================
def main() -> None:
    print_environment_notes()
    output_paths = ensure_output_dirs(CFG)
    save_json(asdict(CFG), output_paths["base"] / "config.json")

    df = load_metadata(CFG)

    print("=" * 80)
    print("DATASET OVERVIEW")
    print("=" * 80)
    print(f"Metadata shape after filtering: {df.shape}")
    print(df["label"].value_counts().sort_index())
    print()

    X, y, ids, lead_names, fs, failed = build_feature_table(df, CFG)
    print("Built V1-V3 feature table")
    print(f"X shape: {X.shape}")
    print(f"y shape: {y.shape}")
    print(f"Lead names original : {lead_names}")
    print(f"Sampling fs         : {fs}")
    print(f"Failed records      : {len(failed)}")
    if failed:
        print("Sample failures:", failed[:5])
    print()

    X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
        X, y, ids,
        test_size=CFG.holdout_size,
        stratify=y,
        random_state=CFG.random_state,
    )

    print("=" * 80)
    print("TRAIN / HOLDOUT SPLIT")
    print("=" * 80)
    print(f"Train size: {X_train.shape}, positives: {int(y_train.sum())}/{len(y_train)}")
    print(f"Test  size: {X_test.shape}, positives: {int(y_test.sum())}/{len(y_test)}")
    print()

    model_spaces = build_model_search_spaces(CFG)
    if not model_spaces:
        raise RuntimeError("No models available to run.")

    results_rows: List[Dict[str, Any]] = []

    for model_name, (estimator, param_dist) in model_spaces.items():
        search = run_random_search(model_name, estimator, param_dist, X_train, y_train, CFG)

        cv_results = pd.DataFrame(search.cv_results_).sort_values(by="rank_test_f1")
        save_dataframe(cv_results, output_paths["metrics"] / f"cv_results_{model_name}.csv")

        print(f"Building OOF predictions for {model_name}...")
        oof_prob = oof_predictions_from_best_estimator(search.best_estimator_, X_train, y_train, CFG)

        threshold_metrics = optimize_threshold(y_train, oof_prob, CFG)
        best_threshold = threshold_metrics["threshold"]

        threshold_df = pd.DataFrame(threshold_metrics["threshold_table"])
        save_dataframe(threshold_df, output_paths["metrics"] / f"threshold_table_{model_name}.csv")

        model_for_holdout = maybe_calibrate_model(search.best_estimator_, X_train, y_train, CFG)
        holdout_metrics = evaluate_on_holdout(model_for_holdout, X_test, y_test, best_threshold)

        detail = {
            "model": model_name,
            "best_params": search.best_params_,
            "oof_threshold_metrics": {k: v for k, v in threshold_metrics.items() if k != "threshold_table"},
            "holdout_metrics": holdout_metrics,
        }
        save_json(detail, output_paths["metrics"] / f"details_{model_name}.json")

        if CFG.save_oof_predictions:
            pred_df = pd.DataFrame({
                "patient_id": ids_train,
                "y_true": y_train,
                "oof_prob": oof_prob,
            })
            save_dataframe(pred_df, output_paths["predictions"] / f"oof_predictions_{model_name}.csv")

        holdout_pred_df = pd.DataFrame({
            "patient_id": ids_test,
            "y_true": y_test,
            "prob": holdout_metrics["probabilities"],
            "pred": holdout_metrics["predictions"],
        })
        save_dataframe(holdout_pred_df, output_paths["predictions"] / f"holdout_predictions_{model_name}.csv")

        results_rows.append({
            "model": model_name,
            "cv_best_f1": float(search.best_score_),
            "oof_threshold": best_threshold,
            "oof_f1": threshold_metrics["f1"],
            "oof_balanced_accuracy": threshold_metrics["balanced_accuracy"],
            "oof_precision": threshold_metrics["precision"],
            "oof_recall": threshold_metrics["recall"],
            "oof_roc_auc": threshold_metrics["roc_auc"],
            "oof_pr_auc": threshold_metrics["pr_auc"],
            "holdout_f1": holdout_metrics["f1"],
            "holdout_balanced_accuracy": holdout_metrics["balanced_accuracy"],
            "holdout_precision": holdout_metrics["precision"],
            "holdout_recall": holdout_metrics["recall"],
            "holdout_roc_auc": holdout_metrics["roc_auc"],
            "holdout_pr_auc": holdout_metrics["pr_auc"],
            "best_params": json.dumps(search.best_params_),
        })

    results_df = pd.DataFrame(results_rows).sort_values(
        by=["holdout_recall", "holdout_f1", "holdout_balanced_accuracy", "holdout_roc_auc"],
        ascending=False,
    )

    save_dataframe(results_df, output_paths["base"] / "final_summary_round2.csv")

    print("\n" + "=" * 80)
    print("FINAL LEADERBOARD - ROUND 2")
    print("=" * 80)
    print(results_df.to_string(index=False))
    print()

    best_row = results_df.iloc[0].to_dict()
    save_json(best_row, output_paths["base"] / "best_model_summary_round2.json")

    print("Best experiment:")
    print(json.dumps(best_row, indent=2))
    print("\nAll results saved under:", output_paths["base"].resolve())


if __name__ == "__main__":
    main()

ENVIRONMENT NOTES
XGBoost available   : True
imblearn available  : True
PyWavelets available: True


DATASET OVERVIEW
Metadata shape after filtering: (356, 5)
label
0    287
1     69
Name: count, dtype: int64

Built V1-V3 feature table
X shape: (356, 126)
y shape: (356,)
Lead names original : ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
Sampling fs         : 100.0
Failed records      : 0

TRAIN / HOLDOUT SPLIT
Train size: (284, 126), positives: 55/284
Test  size: (72, 126), positives: 14/72


===== Running RandomizedSearch for logistic =====
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best params for logistic: {'select__k': 30, 'clf__solver': 'lbfgs', 'clf__penalty': 'l2', 'clf__class_weight': {0: 1, 1: 3}, 'clf__C': 1.0}
Best CV F1 for logistic: 0.5204
Building OOF predictions for logistic...
    OOF fold 1/5 done
    OOF fold 2/5 done
    OOF fold 3/5 done
    OOF fold 4/5 done
    OOF fold 5/5 done

===== Running RandomizedSearch for r